# ARC Cluster Inspector + Annotator

Browse all tasks in a cluster. For each task:
- All training pairs + test pair as colour grids
- LLM-generated description (from `results/cluster_inspection.txt`)
- A textarea to type your own description (saved to `data/human_descriptions.json`)

**Controls:** Save & Next · Skip · ← Prev · jump to any task ID.

Change `CLUSTER` in cell 3 to inspect a different cluster, then re-run cells 3 and 4.


In [ ]:
%matplotlib inline
import json, re
from pathlib import Path
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import ipywidgets as widgets
from IPython.display import display, clear_output

# ARC 10-colour palette
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)

def _find_project_root() -> Path:
    p = Path.cwd()
    for _ in range(8):
        if (p / 'data' / 'training').exists():
            return p
        if (p / 'CLAUDE.md').exists():
            return p
        p = p.parent
    raise FileNotFoundError(
        'Cannot find project root. '
        'Run: jupyter notebook from inside the arc-agi-solver directory.'
    )

PROJECT = _find_project_root()
DESC_FILE = PROJECT / 'data' / 'human_descriptions.json'

human_descs = json.loads(DESC_FILE.read_text()) if DESC_FILE.exists() else {}
print(f'Project root: {PROJECT}')
print(f'Human descriptions already saved: {len(human_descs)}')


In [ ]:
def parse_inspection(path: Path) -> dict:
    """Parse cluster_inspection.txt into {cluster_id: {task_id: description}}."""
    text = path.read_text()
    result = {}
    cluster_pat = re.compile(
        r'={60}\nCluster (\d+) \(n=\d+\)\n={60}\n(.*?)(?=\n={60}\nCluster |\Z)',
        re.DOTALL,
    )
    task_pat = re.compile(
        r'  ([0-9a-f]{8}):\n(.*?)(?=\n  [0-9a-f]{8}:|\Z)',
        re.DOTALL,
    )
    for m in cluster_pat.finditer(text):
        cid = int(m.group(1))
        body = m.group(2)
        tasks = {}
        for tm in task_pat.finditer(body):
            tasks[tm.group(1)] = tm.group(2).strip()
        result[cid] = tasks
    return result

cluster_data = parse_inspection(PROJECT / 'results' / 'cluster_inspection.txt')
print(f'Parsed {len(cluster_data)} clusters: {sorted(cluster_data.keys())}')


In [ ]:
# ── Change CLUSTER to inspect a different cluster ────────────────────────
CLUSTER = 9

task_desc = cluster_data[CLUSTER]
task_ids  = sorted(task_desc.keys())

tasks = {}
for tid in task_ids:
    p = PROJECT / 'data' / 'training' / f'{tid}.json'
    if p.exists():
        tasks[tid] = json.loads(p.read_text())
    else:
        print(f'  Warning: {tid}.json not found')

print(f'Cluster {CLUSTER}: {len(tasks)} / {len(task_ids)} tasks loaded')


In [ ]:
class ClusterAnnotator:

    def __init__(self):
        # Start at first unannotated task in this cluster
        self.idx = next(
            (i for i, tid in enumerate(task_ids) if tid not in human_descs),
            0
        )

        # ── Widgets ───────────────────────────────────────────────────────
        self.fig_out  = widgets.Output()
        self.llm_out  = widgets.Output()

        self.textarea = widgets.Textarea(
            placeholder='Describe the transformation rule in your own words…',
            layout=widgets.Layout(width='100%', height='120px'),
        )
        self.progress = widgets.HTML()
        self.status   = widgets.HTML()

        self.btn_prev = widgets.Button(description='← Prev',
                                       layout=widgets.Layout(width='100px'))
        self.btn_skip = widgets.Button(description='Skip',
                                       layout=widgets.Layout(width='100px'))
        self.btn_save = widgets.Button(description='Save & Next',
                                       button_style='success',
                                       layout=widgets.Layout(width='130px'))
        self.jump_box = widgets.Text(placeholder='task id…',
                                     layout=widgets.Layout(width='160px'))
        self.btn_jump = widgets.Button(description='Go',
                                       layout=widgets.Layout(width='60px'))

        self.btn_prev.on_click(lambda _: self.move(-1))
        self.btn_skip.on_click(lambda _: self.move(+1))
        self.btn_save.on_click(lambda _: self.save_and_next())
        self.btn_jump.on_click(lambda _: self.jump())

        controls = widgets.HBox([
            self.btn_prev, self.btn_skip, self.btn_save,
            widgets.Label('  Jump to:'), self.jump_box, self.btn_jump,
        ])
        ui = widgets.VBox([
            self.progress,
            self.fig_out,
            widgets.HTML('<b>LLM description:</b>'),
            self.llm_out,
            widgets.HTML('<b>Your description:</b>'),
            self.textarea,
            controls,
            self.status,
        ])
        display(ui)
        self.render()

    # ── Rendering ──────────────────────────────────────────────────────────

    def render(self):
        tid  = task_ids[self.idx]
        task = tasks.get(tid)
        done = len(human_descs)
        annotated = ' ✓' if tid in human_descs else ''

        self.progress.value = (
            f'<b>Task {self.idx + 1} / {len(task_ids)}</b> — '
            f'<code>{tid}</code>{annotated} &nbsp;·&nbsp; '
            f'{done} annotated total'
        )
        self.textarea.value = human_descs.get(tid, {}).get('description', '')
        self.status.value = ''

        # LLM description
        with self.llm_out:
            clear_output(wait=True)
            llm_text = task_desc.get(tid, '(no LLM description)')
            print(llm_text)

        # Grids
        with self.fig_out:
            clear_output(wait=True)
            if task is None:
                print(f'{tid} — task file not found')
                return
            pairs = task['train']
            test  = task['test'][0]
            n_rows = len(pairs) + 1
            fig, axes = plt.subplots(n_rows, 2,
                                     figsize=(6, 2.5 * n_rows),
                                     squeeze=False)
            for i, pair in enumerate(pairs):
                for j, (grid, label) in enumerate([
                    (pair['input'],  f'Train {i+1}  input'),
                    (pair['output'], f'Train {i+1}  output'),
                ]):
                    axes[i, j].imshow(np.array(grid), cmap=CMAP, norm=NORM,
                                      interpolation='nearest')
                    axes[i, j].set_title(label, fontsize=8)
                    axes[i, j].axis('off')
            axes[-1, 0].imshow(np.array(test['input']), cmap=CMAP, norm=NORM,
                               interpolation='nearest')
            axes[-1, 0].set_title('Test  input', fontsize=8)
            axes[-1, 0].axis('off')
            if 'output' in test:
                axes[-1, 1].imshow(np.array(test['output']), cmap=CMAP,
                                   norm=NORM, interpolation='nearest')
                axes[-1, 1].set_title('Test  output', fontsize=8)
            else:
                axes[-1, 1].set_title('Test output (hidden)', fontsize=8)
            axes[-1, 1].axis('off')
            plt.tight_layout()
            plt.show()

    # ── Actions ────────────────────────────────────────────────────────────

    def save_and_next(self):
        tid  = task_ids[self.idx]
        desc = self.textarea.value.strip()
        if not desc:
            self.status.value = (
                '<span style="color:orange">⚠ Description is empty — skipping save.</span>'
            )
            self.move(+1)
            return
        human_descs[tid] = {
            'description': desc,
            'timestamp':   datetime.now().isoformat(timespec='seconds'),
        }
        DESC_FILE.write_text(json.dumps(human_descs, indent=2))
        self.status.value = f'<span style="color:green">✓ Saved {tid}</span>'
        self.move(+1)

    def move(self, delta):
        self.idx = max(0, min(len(task_ids) - 1, self.idx + delta))
        self.render()

    def jump(self):
        target = self.jump_box.value.strip()
        if target in task_ids:
            self.idx = task_ids.index(target)
            self.render()
        else:
            self.status.value = (
                f'<span style="color:red">Task {target!r} not in cluster {CLUSTER}</span>'
            )


annotator = ClusterAnnotator()
